# Limpieza del padrón municipal (INE)

En este notebook realizamos la limpieza del dataset de padrón municipal por municipio y año.

## 1. Carga de librerías

In [1]:
import pandas as pd
import numpy as np

## 2. Carga del dataset

In [2]:
# Cargamos el csv
df_padron_raw = pd.read_csv("../data_raw/padron_2003_2022.csv", sep=";", low_memory=False)

## 3. Exploración inicial

In [3]:
# Comprobamos la estructura
df_padron_raw.head()

,Municipios,Sexo,Periodo,Total
0,44001 Ababuj,Total,2025,73
1,44001 Ababuj,Total,2024,74
2,44001 Ababuj,Total,2023,70
3,44001 Ababuj,Total,2022,72
4,44001 Ababuj,Total,2021,76


In [4]:
df_padron_raw.shape

(732420, 4)

In [5]:
# Miramos tipos de datos
df_padron_raw.dtypes

Municipios    object
Sexo          object
Periodo        int64
Total         object
dtype: object

## 4. Normalización de nombres de columnas

In [6]:
# Normalizamos nombres de columnas para consistencia
df_padron_raw.columns = (
    df_padron_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

## 5. Filtrado de población total

In [7]:
# Miramos que valores hay en 'sexo'
df_padron_raw['sexo'].value_counts()

sexo
Total      244140
Hombres    244140
Mujeres    244140
Name: count, dtype: int64

In [8]:
# Nos interesa la población total del municipio, así que filtramos solo por 'Total'
df_padron = df_padron_raw[df_padron_raw['sexo'] == "Total"].copy()

## 6. Conversión de población a númerico

In [9]:
# Convertimos 'total' a númerico ya que nos indica el número de habitantes
df_padron['total'] = (
    df_padron['total']
    .astype(str)
    .str.replace(".", "", regex=False)
)

In [10]:
df_padron['total'] = pd.to_numeric(df_padron['total'], errors="coerce")

## 7. Tratamiento de nulos

### Valores faltantes en población

Al revisar la columna `total` se detectaron algunos valores nulos (NaN).
Estos casos corresponden a municipios y años en los que no hay dato de población
disponible en el dataset original.

Dado que no es posible analizar la evolución de la población sin ese valor,
se decidió eliminar estas observaciones del conjunto de datos.

In [11]:
# Comprobamos si hay NaN en 'total'
df_padron['total'].isna().sum()

np.int64(8764)

In [12]:
# Comprobamos en que años están los NaN
df_padron[df_padron['total'].isna()]['periodo'].value_counts().sort_index()

periodo
1996      42
1997    8138
1998      40
1999      37
2000      34
2001      31
2002      30
2003      30
2004      29
2005      29
2006      28
2007      27
2008      26
2009      26
2010      24
2011      22
2012      22
2013      21
2014      21
2015      19
2016      13
2017      14
2018      14
2019       7
2020       7
2021       7
2022       7
2023       7
2024       6
2025       6
Name: count, dtype: int64

In [13]:
# Eliminaremos las filas con NaN, ya que no podemos analizar población si no existen los datos
df_padron = df_padron.dropna(subset=['total'])

In [14]:
df_padron['total'].isna().sum()

np.int64(0)

## 8. Separación de código y nombre del municipio

La columna `municipios` contiene tanto el código INE del municipio como su nombre.

Para facilitar el análisis y la posterior unión con otros datasets, esta columna
la separamos en dos:
- `codigo_municipio`, que se utilizará como identificador único
- `municipio`, que mantiene el nombre del municipio.

In [15]:
# Vemos cuántos municipios únicos hay
df_padron['municipios'].nunique()

8136

In [16]:
# Separamos el código del nombre del municipio
df_padron[['codigo_municipio', 'municipio']] = df_padron['municipios'].str.split(" ", n=1, expand=True)

In [17]:
# Añadimos la columna nueva al inicio
cols = ['codigo_municipio', 'municipio'] + [col for col in df_padron.columns if col not in ['codigo_municipio', 'municipio']]
df_padron = df_padron[cols]

## 9. Limpieza final del dataset

In [18]:
# Convertimos a string de 5 dígitos
df_padron['codigo_municipio'] = df_padron['codigo_municipio'].astype(str).str.zfill(5)

In [19]:
años_por_municipio = df_padron.groupby("codigo_municipio")["periodo"].nunique()
años_por_municipio.describe()

count    8136.000000
mean       28.930187
std         1.095913
min         2.000000
25%        29.000000
50%        29.000000
75%        29.000000
max        29.000000
Name: periodo, dtype: float64

In [20]:
años_por_municipio.value_counts().sort_index()

periodo
2        1
7        7
9        1
10       6
11       2
12       2
13       1
15       2
16       2
17       2
18       1
19       1
20       3
22       1
24       1
25       3
26       3
27       3
28       2
29    8092
Name: count, dtype: int64

In [21]:
# Eliminamos la columna 'municipios' que era la original antes de separarla
df_padron = df_padron.drop(columns='municipios')

In [22]:
# Podemos eliminar la columna 'sexo' ya que no aporta información después de que hayamos filtrado solo 'Total'
df_padron = df_padron.drop(columns='sexo')

In [23]:
# Comprobamos duplicados municipio - año
df_padron.duplicated(subset=['codigo_municipio', 'periodo']).sum()

np.int64(0)

In [24]:
# Renombramos la columna 'total' para que sea más explicativa
df_padron = df_padron.rename(columns={"total": "poblacion"})

In [25]:
# Ordenamos el dataset 
df_padron = df_padron.sort_values(['codigo_municipio', 'periodo'])

df_padron.head()

,codigo_municipio,municipio,periodo,poblacion
31979,01001,Alegría-Dulantzi,1996,1234.0
31977,01001,Alegría-Dulantzi,1998,1259.0
31976,01001,Alegría-Dulantzi,1999,1329.0
31975,01001,Alegría-Dulantzi,2000,1401.0
31974,01001,Alegría-Dulantzi,2001,1486.0


In [26]:
df_padron = df_padron[["codigo_municipio", "municipio", "periodo", "poblacion"]]

In [27]:
df_padron.duplicated(subset=["codigo_municipio", "periodo"]).sum()

np.int64(0)

In [28]:
df_padron.info()

<class 'pandas.core.frame.DataFrame'>
Index: 235376 entries, 31979 to 374220
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   codigo_municipio  235376 non-null  object 
 1   municipio         235376 non-null  object 
 2   periodo           235376 non-null  int64  
 3   poblacion         235376 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 9.0+ MB


## 10. Exportación del dataset limpio

El dataset final contiene una fila por municipio y año:
`codigo_municipio`, `municipio`, `periodo` (año) y `poblacion`.

In [29]:
# Guardamos el dataset limpio
df_padron.to_csv("../data_processed/padron_limpio.csv", index=False)